# Notebook 08 — Comparativo Final e Transição para o Desafio Final

**Disciplina:** Machine Learning Avançado — FIAP  
**Aula 04:** Métodos Avançados em Reinforcement Learning  
**Pré-requisito:** Notebooks 01 a 07 desta aula  
**Bibliotecas:** numpy, matplotlib (sem gymnasium, sem torch)


| | |
|---|---|
| **Aula** | Aula 04 — Métodos Avançados em Reinforcement Learning |
| **Notebook** | 08 — Comparativo Final |
| **Seções** | 4.1–4.8 |
| **Tempo de leitura** | ~10 min |
| **Tempo de execução** | ~1 min |

**Pré-requisitos:** Notebooks 01–07 desta aula.

**Competências para o Desafio Final:** Selecionar o subcampo adequado para um problema real; justificar a escolha com base nas restrições do problema; planejar a arquitetura de um agente de RL avançado para o Desafio Final.

---

### Recapitulando

A Aula 04 percorreu sete extensões do RL clássico: *model-based* RL (NB02), MARL (NB03), *offline* RL (NB04), hierárquico e imitação (NB05), RLHF (NB06) e segurança (NB07). Este notebook consolida o mapa e apresenta o guia de decisão para orientar o Desafio Final.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import rl_utils
rl_utils.info_versoes()
rl_utils.configurar_matplotlib()
rl_utils.definir_seeds(42)

Biblioteca           Versão
--------------------------------
gymnasium            1.0.0
torch                2.11.0+cu130
numpy                2.4.2
matplotlib           3.10.8
pandas               3.0.1
scikit-learn         1.8.0


## Bloco 1 — Contexto e pergunta central

Os notebooks anteriores desta aula cobriram sete extensões do RL clássico, cada uma respondendo a uma limitação diferente dos métodos das Aulas 02 e 03.

> **Pergunta central:** diante de um problema real, como escolher qual família de métodos avançados aplicar?

Este notebook consolida os sete subcampos em uma visão única e apresenta um guia prático de decisão.


## Bloco 2 — Mini teoria: mapa dos subcampos avançados

Cada subcampo avançado nasce de uma limitação concreta do RL clássico:

| # | Subcampo | Limitação que resolve | Algoritmo representativo |
|---|---|---|---|
| 1 | *Model-based* RL | Agente reage; não planeja | Dyna-Q, MuZero |
| 2 | RL Multiagente (MARL) | Um único agente; ambiente estático | QMIX, MADDPG |
| 3 | *Offline* RL | Requer interação online custosa | CQL, IQL |
| 4 | RL Hierárquico | Horizonte longo; recompensa esparsa | Options, HRL |
| 5 | Imitação / IRL | Recompensa difícil de especificar | BC, GAIL, MaxEnt IRL |
| 6 | RLHF | Preferências humanas como sinal | PPO + *reward model* |
| 7 | Segurança e infraestrutura | Agente otimiza métrica errada | Auditoria, *reward shaping* |

A decisão de qual subcampo usar depende do problema, não do algoritmo preferido.


## Bloco 3 — Código didático

Três visualizações consolidam os 7 subcampos:

1. **Tabela de referência rápida** — nome, problema central, quando usar
2. **Radar chart** — perfil de cada subcampo em 4 dimensões (complexidade de dados, custo computacional, maturidade técnica, amplitude de aplicação)
3. **Guia de decisão** — 7 perguntas sequenciais para selecionar o subcampo certo para um problema real

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Tabela dos subcampos: [nome, problema_central, quando_usar]
subcampos = [
    ("Model-based RL",    "Sem modelo do ambiente",           "Simulação disponível ou dados suficientes para aprender modelo"),
    ("MARL",              "Múltiplos agentes interagem",      "Jogos, mercados, robótica cooperativa"),
    ("Offline RL",        "Interação online impossível",      "Saúde, finanças, sistemas críticos com dataset histórico"),
    ("RL Hierárquico",    "Horizonte longo / recomp. esparsa","Navegação, manipulação com subtarefas decomponíveis"),
    ("Imitação / IRL",    "Recompensa desconhecida",          "Agente deve aprender com demonstrações de especialista"),
    ("RLHF",              "Preferência humana como objetivo", "Geração de texto, assistentes, sistemas alinhados"),
    ("Segurança / Infra", "Risco de reward hacking",         "Qualquer aplicação de alto impacto antes do deploy"),
]

print(f"{'Subcampo':<22} {'Problema central':<36} {'Quando usar'}")
print("─" * 100)
for nome, problema, quando in subcampos:
    print(f"{nome:<22} {problema:<36} {quando}")

Cada subcampo foi avaliado em quatro dimensões (escala ilustrativa 1–5). Use o radar para identificar rapidamente qual método é mais exigente antes de escolhê-lo para um projeto real.

In [ ]:
# Radar chart: perfil de cada subcampo em 4 dimensões
# Escala 1-5 (ilustrativa): complexidade de dados, custo computacional,
# maturidade técnica, amplitude de aplicação

dimensoes = ["Complexidade\nde dados", "Custo\ncomputacional",
             "Maturidade\ntécnica", "Amplitude de\naplicação"]

perfis = {
    "Model-based":  [3, 4, 4, 4],
    "MARL":         [3, 4, 3, 3],
    "Offline RL":   [4, 3, 3, 4],
    "Hierárquico":  [2, 3, 3, 3],
    "Imitação/IRL": [3, 2, 4, 4],
    "RLHF":         [4, 5, 4, 4],
    "Segurança":    [2, 1, 3, 5],
}

N = len(dimensoes)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(2, 4, figsize=(14, 7), subplot_kw=dict(polar=True))
axes = axes.flatten()

cores = plt.cm.tab10(np.linspace(0, 0.9, len(perfis)))

for idx, (nome, vals) in enumerate(perfis.items()):
    ax = axes[idx]
    v = vals + vals[:1]
    ax.plot(angles, v, color=cores[idx], linewidth=2)
    ax.fill(angles, v, color=cores[idx], alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(dimensoes, fontsize=7)
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.set_yticklabels(["1","2","3","4","5"], fontsize=6)
    ax.set_ylim(0, 5)
    ax.set_title(nome, fontsize=9, fontweight="bold", pad=10)

axes[-1].axis("off")
fig.suptitle("Perfil dos subcampos avançados de RL\n(escala ilustrativa 1–5)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("nb08_radar_subcampos.png", dpi=100, bbox_inches="tight")
plt.show()
print("Gráfico salvo: nb08_radar_subcampos.png")


Gráfico salvo: nb08_radar_subcampos.png


### Leitura rápida do radar

Cada eixo representa uma dimensão de complexidade (escala 1–5):

- **RLHF** e **Offline RL** têm os perfis mais "inflados" — maior exigência de dados e computação, mas também maior amplitude de aplicação. São os subcampos com mais demanda em sistemas de IA em produção hoje.
- **Segurança e Infraestrutura** tem o menor custo (baixa computação — são práticas, não novos algoritmos) com a maior amplitude: aplica-se a **todos** os outros subcampos antes do deploy.
- **RL Hierárquico** e **Imitação/IRL** têm maturidade técnica intermediária — áreas ainda ativas de pesquisa, com algoritmos prontos para uso (BC, Options) mas sem consenso estabelecido nas fronteiras.

Use o radar para orientar a escolha: se o problema exige o subcampo de maior custo, verifique primeiro se há uma solução mais simples (RL clássico das Aulas 02–03) que resolve o problema antes de escalar.

In [ ]:
# Guia de decisão: fluxo textual estruturado
perguntas = [
    ("Tem acesso a um modelo do ambiente ou pode aprendê-lo?",
     "→ Considere Model-based RL (Dyna-Q, MuZero)"),
    ("Há múltiplos agentes interagindo no mesmo ambiente?",
     "→ Considere MARL (QMIX, MADDPG, CTDE)"),
    ("Interação online é impossível — só tem dados históricos?",
     "→ Considere Offline RL (CQL, IQL)"),
    ("O problema tem subtarefas ou horizonte temporal muito longo?",
     "→ Considere RL Hierárquico (Options framework, HRL)"),
    ("A recompensa é difícil de especificar, mas há demonstrações?",
     "→ Considere Imitação / IRL (BC, GAIL, MaxEnt IRL)"),
    ("O objetivo é capturar preferências humanas (ex.: LLMs)?",
     "→ Considere RLHF (PPO + reward model)"),
    ("Vai fazer deploy em contexto de alto impacto?",
     "→ Aplique práticas de Segurança e Infraestrutura antes"),
]

print("GUIA DE DECISÃO — Qual subcampo avançado usar?")
print("=" * 60)
for i, (pergunta, recomendacao) in enumerate(perguntas, 1):
    print(f"\n{i}. {pergunta}")
    print(f"   {recomendacao}")
print("\n(As condições não são mutuamente exclusivas;")
print(" problemas reais frequentemente combinam múltiplos subcampos.)")


GUIA DE DECISÃO — Qual subcampo avançado usar?

1. Tem acesso a um modelo do ambiente ou pode aprendê-lo?
   → Considere Model-based RL (Dyna-Q, MuZero)

2. Há múltiplos agentes interagindo no mesmo ambiente?
   → Considere MARL (QMIX, MADDPG, CTDE)

3. Interação online é impossível — só tem dados históricos?
   → Considere Offline RL (CQL, IQL)

4. O problema tem subtarefas ou horizonte temporal muito longo?
   → Considere RL Hierárquico (Options framework, HRL)

5. A recompensa é difícil de especificar, mas há demonstrações?
   → Considere Imitação / IRL (BC, GAIL, MaxEnt IRL)

6. O objetivo é capturar preferências humanas (ex.: LLMs)?
   → Considere RLHF (PPO + reward model)

7. Vai fazer deploy em contexto de alto impacto?
   → Aplique práticas de Segurança e Infraestrutura antes

(As condições não são mutuamente exclusivas;
 problemas reais frequentemente combinam múltiplos subcampos.)


## Bloco 4 — Interpretação pedagógica

### O que o radar revela

- **RLHF** e **Offline RL** exigem mais dados e têm maior amplitude de aplicação — são os subcampos com mais demanda em produção hoje.
- **Segurança e infraestrutura** tem custo computacional baixo mas amplitude máxima: toda aplicação se beneficia de boas práticas de auditoria.
- **RL Hierárquico** e **Imitação/IRL** têm maturidade técnica média — áreas ainda ativas de pesquisa.

### Os sete subcampos como extensões coerentes

Cada subcampo mantém o núcleo do RL (agente, ambiente, recompensa, política) e relaxa **uma** restrição:

| Restrição relaxada | Subcampo habilitado |
|---|---|
| "O agente não conhece o modelo" | *Model-based* RL |
| "Há um único agente" | MARL |
| "O agente interage online" | *Offline* RL |
| "As decisões são atômicas" | RL Hierárquico |
| "A recompensa é dada" | Imitação / IRL |
| "A recompensa é objetiva" | RLHF |
| "A métrica é confiável" | Segurança |

Conhecer o mapa é saber qual ferramenta pegar diante de um problema novo.


## Autoavaliação

Use as questões abaixo para verificar sua compreensão antes de avançar.

<details>
<summary><strong>Questão 1.</strong> O guia de decisão do Bloco 3 apresenta 7 perguntas para selecionar o subcampo. Aplique o guia a este cenário: "Uma empresa de logística quer treinar um agente para otimizar rotas de entrega em uma frota de 50 veículos. Os veículos precisam coordenar para evitar congestionamentos, e o histórico de 2 anos de rotas e tempos de entrega está disponível. Novas simulações são caras." Quais subcampos são ativados?</summary>

**Resposta:** O cenário ativa múltiplos subcampos simultaneamente:

1. **MARL** — 50 veículos aprendendo ao mesmo tempo, com interdependência (um veículo escolhendo uma rota afeta o congestionamento para os outros).
2. **Offline RL** — o histórico de 2 anos existe, e simulações novas são caras; o aprendizado deve aproveitar os dados históricos ao máximo.
3. **Segurança e Infraestrutura** — deploy em produção com frota real; efeitos colaterais de uma política ruim têm custo real.

Opcionalmente: **RL Hierárquico** se houver subtarefas claras (decidir qual área da cidade cobrir, depois a rota específica).

Essa é a natureza de problemas reais: raramente um único subcampo é suficiente.
</details>

<details>
<summary><strong>Questão 2.</strong> O Bloco 4 apresenta os 7 subcampos como "restrições relaxadas" do RL clássico. Por que essa perspectiva é pedagogicamente mais útil do que simplesmente listar os algoritmos de cada subcampo?</summary>

**Resposta:** Listar algoritmos responde "o que existe" — mas não responde "quando usar" nem "por que funciona". A perspectiva de "restrição relaxada" conecta diretamente o **problema** ao **subcampo**, sem passar pela lista de algoritmos.

Com essa estrutura, ao encontrar um problema novo, o processo de raciocínio é:
1. Quais suposições do RL clássico esse problema viola?
2. Qual subcampo remove essa suposição?
3. Quais algoritmos desse subcampo são mais maduros?

Isso é mais robusto do que memorizar algoritmos: novos algoritmos surgem constantemente, mas as **restrições fundamentais** dos problemas são estáveis. A taxonomia por restrição relaxada continua válida quando MuZero é substituído por algo melhor — o subcampo *model-based* ainda existe.
</details>

<details>
<summary><strong>Questão 3.</strong> O Bloco 5 afirma que "os métodos das Aulas 02 e 03 continuam sendo a base — os subcampos avançados entram quando o problema exige". Dê um exemplo concreto de quando seria um erro escolher um subcampo avançado quando um método clássico seria suficiente.</summary>

**Resposta (exemplos válidos):**

- **Usar Offline RL** para treinar um agente em um simulador rápido e gratuito (ex.: CartPole). O simulador permite interação online ilimitada — Offline RL resolve um problema (ausência de interação) que não existe aqui. Q-Learning ou PPO convergem mais rápido e são mais simples de depurar.

- **Usar MARL** para um problema onde há múltiplos agentes, mas eles são independentes (não interagem). Se os 50 veículos de logística operassem em cidades diferentes sem nunca competir por recursos, cada agente poderia ser treinado independentemente com PPO — MARL adiciona complexidade sem benefício.

- **Usar RLHF** quando a recompensa já está bem definida e quantificável (ex.: minimizar tempo de entrega em segundos). RLHF substitui uma recompensa explícita por preferências humanas — mas se a recompensa explícita já captura o objetivo, a substituição só adiciona ruído e custo de anotação.

O princípio: use o método mais simples que resolve o problema. Subcampos avançados têm custo de complexidade real.
</details>

## Mapeamento para o Desafio Final

Este notebook fecha a Aula 04. A tabela abaixo mapeia cada etapa do Desafio Final ao subcampo avançado que a suporta — e ao notebook onde as competências foram construídas.

| Etapa do Desafio Final | Subcampo relevante | Notebook de referência |
|---|---|---|
| Definir o problema e suas restrições | Guia de decisão (7 perguntas) | NB01, NB08 (este) |
| Escolher fonte de dados ou ambiente | Offline RL vs interação online | NB04 |
| Coordenar múltiplos agentes ou subtarefas | MARL ou RL Hierárquico | NB03, NB05 |
| Especificar a recompensa | RLHF / IRL quando a recompensa é difícil | NB05, NB06 |
| Auditar, documentar e garantir reprodutibilidade | Segurança e infraestrutura | NB07 |

**Fluxo de decisão em 4 perguntas:**

1. **Dados históricos ou interação online?** → Somente dados: *Offline RL*. Pode interagir: continua.
2. **Um único agente ou múltiplos?** → Múltiplos com interdependência: *MARL*. Horizonte longo: *Hierárquico*. Um agente simples: continua.
3. **Recompensa explícita e bem definida?** → Sim: PPO / SAC / DQN (Aulas 02–03). Não: *RLHF* ou *IRL*.
4. **Contexto de alto impacto?** → Sempre: aplique as boas práticas de segurança e infraestrutura do NB07 antes do deploy.

Os algoritmos das Aulas 02 e 03 são a base. Os subcampos avançados entram quando o problema viola as suposições do RL clássico — use o mapa acima para identificar qual suposição está sendo relaxada.

## Bloco 5 — Limites e transição para o Desafio Final

### Limites desta aula

- Cada subcampo foi coberto em profundidade didática, não de produção. Implementações completas (MuZero, QMIX, CQL completo) requerem infraestrutura e datasets além do escopo de um notebook.
- Os exemplos usaram ambientes simples (GridWorld, FrozenLake, CartPole). Problemas reais são mais complexos e frequentemente combinam múltiplos subcampos.
- RLHF e segurança, em contextos de LLMs reais, requerem modelos de linguagem base — não cobertos aqui.

### Transição para o Desafio Final (Aula 05)

O **Desafio Final** pede a implementação de um agente de RL aplicado a um problema à sua escolha. Use este mapa para orientar a decisão:

1. **Defina o problema** antes de escolher o algoritmo.
2. **Identifique as restrições**: dados disponíveis, custo de interação, especificação de recompensa.
3. **Selecione o subcampo** com base no guia de decisão acima.
4. **Documente** as escolhas: por que este subcampo? Quais limitações você aceita?

Os métodos das Aulas 02 e 03 (Q-Learning, DQN, PPO, SAC) continuam sendo a base — os subcampos avançados entram quando o problema exige.


In [ ]:
# Glossário dos termos técnicos deste notebook
rl_utils.exibir_glossario([
    'model-based RL', 'offline RL', 'MARL', 'RLHF',
    'IRL', 'hierarchical RL', 'reward hacking', 'alignment', 'behavior cloning',
])

Termo (EN)         Tradução (PT)                Descrição
----------------------------------------------------------------------------------------------------------
IRL                IRL                          Inverse RL — infere a função de recompensa a partir de demonstrações.
MARL               MARL                         Multi-Agent RL — múltiplos agentes aprendendo em um ambiente compartilhado.
RLHF               RLHF                         RL from Human Feedback — usa preferências humanas para aprender recompensa.
alignment          alinhamento                  Garantia de que o agente persegue os objetivos pretendidos pelo designer.
behavior cloning   clonagem comportamental      Imita um especialista via aprendizado supervisionado a partir de demonstrações.
hierarchical RL    RL hierárquico               Decomposição de tarefas em políticas de alto e baixo nível.
model-based RL     RL baseado em modelo         Aprendizado que usa um modelo do ambiente para planejamento.
of

## Leituras e referências

- Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2ª ed.). MIT Press. Capítulos 8, 17.  
  Disponível em: http://incompleteideas.net/book/the-book-2nd.html

- Levine, S. et al. (2020). Offline Reinforcement Learning: Tutorial, Review, and Perspectives. *arXiv:2005.01643*.  
  Disponível em: https://arxiv.org/abs/2005.01643

- Ouyang, L. et al. (2022). Training language models to follow instructions with human feedback. *NeurIPS 2022*.  
  Disponível em: https://arxiv.org/abs/2203.02155

- Amodei, D. et al. (2016). Concrete Problems in AI Safety. *arXiv:1606.06565*.  
  Disponível em: https://arxiv.org/abs/1606.06565

- Lowe, R. et al. (2017). Multi-Agent Actor-Critic for Mixed Cooperative-Competitive Environments. *NeurIPS 2017*.  
  Disponível em: https://arxiv.org/abs/1706.02275

- Spinning Up in Deep RL — OpenAI. Acesso em: abr. 2026.  
  Disponível em: https://spinningup.openai.com
